# AcoustiCare — Fine-Tuning BEATs pada Dataset MIMII (Production-Grade, Anti-Overfitting)

Notebook ini melatih ulang (fine-tune) sebagian lapisan backbone **BEATs** memakai
dataset **MIMII** (`-6_dB_fan`, machine IDs `id_00, id_02, id_04, id_06`), lalu
menghasilkan checkpoint `.pt` yang **drop-in compatible** dengan cell "Load BEATs Model"
di `acousticare_v4_beats.ipynb` — tinggal ganti `CHECKPOINT_FILENAME`.

**Kenapa fine-tuning ini dirancang supaya tidak overfit:**

| Teknik | Penjelasan |
|---|---|
| **Partial freeze** | Hanya beberapa layer transformer teratas (`n_unfrozen_layers`, default 2 dari 12) yang dilatih; sisanya tetap beku dari pretraining AudioSet. Parameter yang bisa dilatih jauh lebih sedikit → risiko overfit jauh lebih kecil dibanding fine-tune penuh. |
| **Held-out machine ID** | Satu unit mesin (`holdout_machine_id`, default `id_06`) **sama sekali tidak pernah dilihat** saat training/validasi — dipakai di akhir sebagai ukuran generalisasi ke unit baru yang jujur (bukan angka yang di-"intip" model). |
| **Split di level klip, bukan window** | Train/val dipisah per **klip audio**, baru di-window — supaya window-window dari klip yang sama (yang saling overlap/mirip) tidak bocor antara train dan val. |
| **Early stopping** | Training berhenti begitu `val_auc` berhenti membaik (`finetune_patience` epoch), lalu backbone dikembalikan ke state dengan `val_auc` terbaik — bukan state di epoch terakhir. |
| **Balanced batch sampler + augmentasi ekstra di kelas minoritas** | MIMII secara alami timpang (jauh lebih banyak klip normal daripada abnormal). `WeightedRandomSampler` membuat proporsi tiap batch ~50/50, dan kelas abnormal diberi **lebih banyak varian augmentasi** (`aug_n_abnormal` > `aug_n_normal`) supaya saat di-oversample, sampelnya tetap bervariasi — bukan mengulang window persis sama yang justru bikin model menghafal beberapa klip abnormal saja. |
| **Window non-overlap saat training** | Deployment pakai hop 1s (window saling overlap 50%) supaya trajektori padat; training pakai hop = window (tanpa overlap) supaya sampel antar-window lebih independen, tidak inflate jumlah sampel dengan duplikat yang sangat mirip. |
| **LR kecil + weight decay + dropout** | `AdamW` dengan LR berbeda untuk backbone (kecil, `2e-5`) vs head baru (lebih besar, `1e-3`), plus weight decay dan dropout di head. |

**Upgrade "Production-Grade Research Code" pada revisi ini:**

| Area | Sebelum | Sesudah |
|---|---|---|
| Loss function | `BCEWithLogitsLoss` | `FocalLoss` custom, dikombinasikan dengan sengaja terhadap `WeightedRandomSampler` yang sudah ada (lihat catatan desain di sel Focal Loss) |
| Metrik evaluasi | AUC saja | AUC + Precision + Recall + F1 (threshold 0.5) tiap akhir epoch |
| LR schedule | Konstan sepanjang training | Linear warmup → Cosine annealing, di-step per batch, per param-group (backbone vs head) |
| Eval/inferensi | `torch.no_grad()`, dan validasi tanpa context manager sama sekali (bug lama — graph tetap dibangun saat validasi) | `torch.inference_mode()` konsisten di semua jalur non-training, termasuk validasi |
| Memori GPU | Tidak dikelola eksplisit | `torch.cuda.empty_cache()` di akhir tiap epoch (kalau CUDA tersedia) |
| Normalisasi amplitudo | Peak-normalize dasar (`x / max(abs(x))`) | `peak_normalize()` yang eksplisit menghilangkan DC offset, meng-guard silence (div-by-zero) dan NaN/Inf, dipakai konsisten di `condition()` maupun `augment()` |
| Checkpoint | `state_dict()` disimpan apa adanya (device asal training) | Dipindah eksplisit ke CPU + `float32` sebelum disimpan, diverifikasi ulang saat reload — dijamin kompatibel di lingkungan FastAPI CPU-only tanpa `map_location` eksplisit |
| Type hints | Tidak ada | Ditambahkan di seluruh fungsi/class inti |

**Catatan skala & waktu:** file `-6_dB_fan.zip` di Zenodo berukuran **~10.9 GB**. Unduhan
dan ekstraksi bisa memakan waktu cukup lama tergantung koneksi Colab. Notebook ini
membatasi jumlah klip per (machine_id, label) lewat `max_clips_per_class_per_id` supaya
waktu training tetap masuk akal dalam satu sesi Colab — naikkan nilainya kalau
sesi/GPU Anda kuat dan ingin memakai lebih banyak data.

**Jalankan dengan runtime GPU** (Runtime → Change runtime type → GPU) — fine-tuning
transformer di CPU jauh lebih lambat.

Seluruh logika di notebook ini (split anti-leakage, freeze/unfreeze, sampler seimbang,
focal loss, scheduler warmup+cosine, training loop, early stopping, format checkpoint)
sudah diuji cell-per-cell secara terisolasi di sandbox terpisah (unit test numerik untuk
`peak_normalize`, `FocalLoss`, scheduler, dan proses save/reload checkpoint CPU-safe) —
hanya unduhan MIMII asli dari Zenodo dan runtime GPU Colab yang tidak bisa diuji dari sana.


## 0. Instalasi & Persiapan Environment

In [ ]:
# Dependency non-torch. torch/torchvision/torchaudio SENGAJA tidak di-install/upgrade
# di sini -- itu sudah disiapkan sekali di venv proyek dengan build CUDA yang benar
# (lihat CONFIG.md). `pip install torch --upgrade` tanpa --index-url akan diam-diam
# mengganti build CUDA yang sudah terpasang dengan build CPU-only default PyPI,
# mematikan GPU tanpa pesan error apa pun -- bug yang ada di revisi sebelumnya.
# %pip install -q onedrivedownloader soundfile

import os


In [ ]:
# ==========================================================
# Checkpoint BEATs pretrained (titik awal fine-tuning)
# ==========================================================
from onedrivedownloader import download

ONEDRIVE_LINK = "https://1drv.ms/u/s!AqeByhGUtINrgcpvdNz8-aYim60CIg?e=53V8pg"
BASE_CHECKPOINT = "BEATs_iter3_plus_AS20K.pt"

if os.path.exists(BASE_CHECKPOINT) and os.path.getsize(BASE_CHECKPOINT) > 10_000_000:
    print(f"Checkpoint sudah ada: {BASE_CHECKPOINT}")
else:
    try:
        download(ONEDRIVE_LINK, filename=BASE_CHECKPOINT, unzip=False)
        print("Checkpoint berhasil diunduh:", BASE_CHECKPOINT)
    except Exception as e:
        print("Gagal unduh otomatis dari OneDrive:", repr(e))
        print("Silakan upload checkpoint (.pt) secara manual.")
        # Import google.colab dijaga terpisah: di luar Colab ini melempar
        # ModuleNotFoundError, yang sebelumnya tidak ditangkap sama sekali dan
        # bikin cell gagal dengan traceback membingungkan alih-alih pesan jelas.
        try:
            from google.colab import files
        except ImportError:
            raise RuntimeError(
                f"Unduhan otomatis gagal dan ini bukan lingkungan Colab. "
                f"Taruh file checkpoint secara manual sebagai '{BASE_CHECKPOINT}' "
                f"di direktori kerja notebook ini, lalu jalankan ulang cell ini."
            ) from e
        up = files.upload()
        pt_files = [f for f in up.keys() if f.lower().endswith(".pt")]
        if not pt_files:
            raise RuntimeError("Tidak ada file .pt yang di-upload.")
        if pt_files[0] != BASE_CHECKPOINT:
            os.rename(pt_files[0], BASE_CHECKPOINT)

assert os.path.exists(BASE_CHECKPOINT), "Checkpoint BEATs pretrained tidak ditemukan."


## 1. Unduh Dataset MIMII (`-6_dB_fan`, ~10.9 GB)

Sumber resmi: [Zenodo record 3384388](https://zenodo.org/records/3384388). Sel ini
idempotent — kalau zip/folder sudah ada, unduhan/ekstraksi dilewati.

**Alternatif lebih cepat:** kalau Anda sudah punya datasetnya di Google Drive, mount
Drive dan langsung set `MIMII_ROOT` ke path di sana, lalu lewati sel unduhan ini —
jauh lebih cepat daripada unduh ulang 10.9 GB setiap sesi Colab baru.


In [ ]:
# MIMII_ROOT = "mimii_data"  # ganti ke path Google Drive Anda kalau sudah punya datanya
# MIMII_ZIP_URL = "https://zenodo.org/records/3384388/files/-6_dB_fan.zip?download=1"
# MIMII_ZIP_NAME = "-6_dB_fan.zip"

# os.makedirs(MIMII_ROOT, exist_ok=True)
# extracted_marker = os.path.join(MIMII_ROOT, "-6_dB_fan")

# if os.path.isdir(extracted_marker):
#     print(f"Dataset sudah terekstrak di {extracted_marker} — lewati unduhan.")
# else:
#     if not os.path.exists(MIMII_ZIP_NAME):
#         print("Mengunduh -6_dB_fan.zip (~10.9 GB) dari Zenodo — ini bisa makan waktu lama...")
#         os.system(f"wget -c -q --show-progress '{MIMII_ZIP_URL}' -O {MIMII_ZIP_NAME}")
#     assert os.path.exists(MIMII_ZIP_NAME) and os.path.getsize(MIMII_ZIP_NAME) > 1_000_000_000, \
#         "Unduhan MIMII gagal atau tidak lengkap (file terlalu kecil / tidak ada)."
#     print("Mengekstrak zip...")
#     os.system(f"unzip -q -o {MIMII_ZIP_NAME} -d {MIMII_ROOT}")

# assert os.path.isdir(extracted_marker), f"Struktur folder tidak sesuai harapan: {extracted_marker} tidak ditemukan."
# print("Dataset siap di:", extracted_marker)

import os

MIMII_ROOT = "Dataset"

assert os.path.isdir(MIMII_ROOT), f"Folder dataset tidak ditemukan: {MIMII_ROOT}"
assert os.path.isdir(os.path.join(MIMII_ROOT, "normal")), "Folder normal tidak ditemukan."
assert os.path.isdir(os.path.join(MIMII_ROOT, "abnormal")), "Folder abnormal tidak ditemukan."

print("Dataset siap di:", MIMII_ROOT)

Dataset siap di: Dataset


## 2. Konfigurasi

In [ ]:
import warnings, copy
from dataclasses import dataclass, replace
from pathlib import Path
from typing import Optional, List, Dict, Tuple, Callable
import numpy as np
import pandas as pd
import scipy.signal as sig
import torch
import torch.nn as nn
import torch.nn.functional as F
import torchaudio
from torch import Tensor
from torch.utils.data import DataLoader, Dataset
from sklearn.metrics import roc_auc_score, precision_score, recall_score, f1_score
from sklearn.model_selection import train_test_split

warnings.filterwarnings("ignore")

@dataclass
class Config:
    sr: int = 16000
    min_duration_sec: float = 2.0
    max_clipping_ratio: float = 0.001
    min_snr_db: float = -10.0
    silence_rms: float = 1e-4
    bp_low: float = 50.0
    bp_high: float = 7500.0
    bp_order: int = 4
    notch_low: float = 300.0
    notch_high: float = 3400.0
    notch_order: int = 2
    window_sec: float = 2.0
    hop_sec: float = 1.0  # dipakai di deployment (v4); training pakai hop=window_sec (non-overlap)

    dataset_root: str = MIMII_ROOT
    snr_level: str = "-6_dB"
    machine_type: str = "fan"

    # unit yang di-hold-out total (tidak pernah dilihat saat train/val) untuk
    # ukuran generalisasi yang jujur terhadap unit baru
    holdout_machine_id: str = "id_06"
    # batasi jumlah klip per (machine_id,label) supaya training tetap tuntas
    # dalam waktu wajar -- naikkan kalau resource Colab Anda kuat
    max_clips_per_class_per_id: int = 200
    train_val_split_frac: float = 0.2

    n_unfrozen_layers: int = 2
    head_dropout: float = 0.3
    finetune_lr_backbone: float = 2e-5
    finetune_lr_head: float = 1e-3
    finetune_weight_decay: float = 1e-2
    finetune_batch_size: int = 16
    finetune_epochs_max: int = 15
    finetune_patience: int = 3

    # Focal Loss (lihat sel "Loss Function" untuk penjelasan desain lengkap).
    # alpha=0.5 = netral secara sengaja, karena keseimbangan kelas sudah ditangani
    # WeightedRandomSampler di bawah -- gamma tetap berperan menekan easy-examples.
    focal_alpha: float = 0.5
    focal_gamma: float = 2.0

    # LR scheduler: proporsi step training yang dipakai untuk linear warmup
    # sebelum masuk fase cosine annealing.
    finetune_warmup_ratio: float = 0.1

    # Augmentasi: kelas minoritas (abnormal, biasanya jauh lebih sedikit di MIMII)
    # dapat lebih banyak varian supaya saat di-oversample lewat sampler, sampelnya
    # tetap bervariasi -- bukan literal mengulang window yang sama persis.
    aug_n_normal: int = 1
    aug_n_abnormal: int = 4
    aug_pitch_range: float = 2.0
    aug_gain_range_db: float = 6.0
    aug_noise_snr: Tuple[float, float] = (10.0, 30.0)

    seed: int = 42

    def bp_clamped(self, fs: int) -> Tuple[float, float]:
        nyq = 0.5 * fs
        return max(1.0, min(self.bp_low, nyq * 0.98)), min(self.bp_high, nyq * 0.99)

    def notch_clamped(self, fs: int) -> Tuple[float, float]:
        nyq = 0.5 * fs
        return max(1.0, min(self.notch_low, nyq * 0.95)), min(self.notch_high, nyq * 0.98)

CFG = Config()
np.random.seed(CFG.seed)
torch.manual_seed(CFG.seed)
print(f"Config siap. dataset_root={CFG.dataset_root}, holdout={CFG.holdout_machine_id}")

Config siap. dataset_root=Dataset, holdout=id_06


## 3. Blok A — Quality Gate, Conditioning, Windowing, Augmentasi

Perubahan pada revisi ini: normalisasi amplitudo puncak (0 dBFS) dipisah menjadi fungsi
`peak_normalize()` tersendiri yang secara eksplisit robust terhadap tiga hal yang bisa
merusak normalisasi di lapangan — (a) **DC offset** dari mikrofon murahan yang menggeser
puncak secara artifisial, (b) **sinyal nyaris hening** yang bisa memicu pembagian dengan
bilangan mendekati nol, dan (c) **NaN/Inf** yang kadang muncul dari resample atau filter
yang tak stabil pada input yang sangat pendek. Fungsi ini yang sekarang dipakai konsisten
baik di `condition()` (jalur utama) maupun `augment()` (supaya varian augmentasi juga
selalu berada di skala 0 dBFS yang sama, bukan cuma jalur asli).


In [ ]:
def estimate_snr(x: np.ndarray, fs: int) -> float:
    fl = max(int(0.05 * fs), 32)
    nf = max(len(x) // fl, 1)
    rms = np.sqrt(np.mean(x[:nf * fl].reshape(nf, fl) ** 2, axis=1) + 1e-12)
    return float(20 * np.log10((np.percentile(rms, 90) + 1e-12) / (np.percentile(rms, 10) + 1e-12)))


def quality_gate(x: np.ndarray, fs: int, cfg: Config) -> Tuple[bool, List[str], float]:
    reasons: List[str] = []
    if len(x) / fs < cfg.min_duration_sec:
        reasons.append("short")
    rms = float(np.sqrt(np.mean(x ** 2) + 1e-12))
    if rms < cfg.silence_rms:
        reasons.append("silent")
    cr = float(np.mean(np.abs(x) >= 0.999)) if len(x) else 1.0
    if cr > cfg.max_clipping_ratio:
        reasons.append("clipping")
    snr = estimate_snr(x, fs) if len(x) > 100 else -99.0
    if snr < cfg.min_snr_db:
        reasons.append("low_snr")
    return len(reasons) == 0, reasons, snr


def bandpass(x: np.ndarray, fs: int, cfg: Config) -> np.ndarray:
    lo, hi = cfg.bp_clamped(fs)
    nyq = 0.5 * fs
    b, a = sig.butter(cfg.bp_order, [lo / nyq, hi / nyq], btype="band")
    return sig.filtfilt(b, a, x).astype(np.float64)


def privacy_notch(x: np.ndarray, fs: int, cfg: Config) -> np.ndarray:
    lo, hi = cfg.notch_clamped(fs)
    nyq = 0.5 * fs
    b, a = sig.butter(cfg.notch_order, [lo / nyq, hi / nyq], btype="bandstop")
    return sig.filtfilt(b, a, x).astype(np.float64)


def peak_normalize(x: np.ndarray, eps: float = 1e-12) -> np.ndarray:
    """Normalisasi amplitudo puncak ke 0 dBFS (peak amplitude -> 1.0).

    Dirancang tahan terhadap tiga kegagalan lapangan yang paling umum pada rekaman
    mikrofon smartphone:
      1. DC offset (bias tegangan dari hardware mic) -- dihilangkan dulu sebelum
         menghitung puncak, supaya offset tidak ikut "dianggap" sinyal.
      2. Sinyal nyaris hening -- dijaga dengan `eps`, tidak dipaksa dibagi bilangan
         mendekati nol yang bisa meledakkan noise lantai jadi amplitudo penuh.
      3. NaN/Inf -- disaring lebih dulu (bisa muncul dari resample/filter pada input
         yang sangat pendek atau tak stabil secara numerik).

    Karena target selalu peak == 1.0 (0 dBFS) apa pun gain aslinya, ini yang membuat
    dua rekaman dari HP dengan sensitivitas mic berbeda berakhir di skala yang sama --
    properti kunci untuk generalisasi lintas-merk smartphone.
    """
    x = np.nan_to_num(x, nan=0.0, posinf=0.0, neginf=0.0)
    x = x - np.mean(x)
    peak = np.max(np.abs(x))
    if peak < eps:
        return x
    return (x / peak).astype(np.float64)


def condition(x_raw: np.ndarray, fs: int, cfg: Config) -> np.ndarray:
    x_raw = np.asarray(x_raw)
    x = x_raw.astype(np.float64)
    if np.issubdtype(x_raw.dtype, np.integer):
        x = x / np.iinfo(x_raw.dtype).max
    x = bandpass(x, fs, cfg)
    x = privacy_notch(x, fs, cfg)
    return peak_normalize(x)


def make_windows(x: np.ndarray, fs: int, cfg: Config) -> List[np.ndarray]:
    win = int(cfg.window_sec * fs)
    hop = int(cfg.hop_sec * fs)
    if win <= 0 or len(x) < win:
        return []
    windows: List[np.ndarray] = []
    start = 0
    while start + win <= len(x):
        w = x[start:start + win]
        ok, _, _ = quality_gate(w, fs, cfg)
        if ok:
            windows.append(w)
        start += hop
    return windows


def aug_pitch(x: np.ndarray, fs: int, semi: float) -> np.ndarray:
    ratio = 2.0 ** (semi / 12.0)
    new_len = int(len(x) / ratio)
    if new_len < 10:
        return x.copy()
    xs = sig.resample(x, new_len)
    if len(xs) >= len(x):
        return xs[:len(x)]
    out = np.zeros_like(x)
    out[:len(xs)] = xs
    return out


def aug_gain(x: np.ndarray, db: float) -> np.ndarray:
    return x * 10.0 ** (db / 20.0)


def aug_noise(x: np.ndarray, snr_db: float, rng: np.random.RandomState) -> np.ndarray:
    sp = np.mean(x ** 2) + 1e-12
    npw = sp / 10.0 ** (snr_db / 10.0)
    return x + rng.normal(0, np.sqrt(npw), len(x))


def augment(
    x: np.ndarray, fs: int, cfg: Config, rng: np.random.RandomState, n: int
) -> List[np.ndarray]:
    variants: List[np.ndarray] = []
    for _ in range(n):
        xa = x.copy()
        xa = aug_pitch(xa, fs, rng.uniform(-cfg.aug_pitch_range, cfg.aug_pitch_range))
        xa = aug_gain(xa, rng.uniform(-cfg.aug_gain_range_db, cfg.aug_gain_range_db))
        xa = aug_noise(xa, rng.uniform(*cfg.aug_noise_snr), rng)
        xa = peak_normalize(xa)
        variants.append(xa)
    return variants

print("Blok A ready.")

Blok A ready.


## 4. Manifest MIMII (path, machine_id, label)

In [ ]:
# def discover_ids(cfg: Config) -> List[str]:
#     d = Path(cfg.dataset_root) / f"{cfg.snr_level}_{cfg.machine_type}" / cfg.machine_type
#     return sorted([p.name for p in d.iterdir() if p.is_dir() and p.name.startswith("id_")]) if d.exists() else []


# def list_wavs(cfg: Config, mid: str, label: str) -> List[str]:
#     d = Path(cfg.dataset_root) / f"{cfg.snr_level}_{cfg.machine_type}" / cfg.machine_type / mid / label
#     d = Path(cfg.dataset_root)
#     return sorted(str(p) for p in d.glob("*.wav")) if d.exists() else []


# def build_manifest(cfg: Config) -> pd.DataFrame:
#     rows = []
#     for mid in discover_ids(cfg):
#         for lab in ("normal", "abnormal"):
#             for p in list_wavs(cfg, mid, lab):
#                 rows.append({"path": p, "model_id": mid, "label": lab})
#     return pd.DataFrame(rows, columns=["path", "model_id", "label"])


# MF = build_manifest(CFG)
# print(f"Manifest: {len(MF)} file. Machine IDs: {discover_ids(CFG)}")
# assert len(MF) > 0, (
#     "Manifest kosong. Cek CFG.dataset_root dan struktur folder "
#     "(<root>/-6_dB_fan/fan/id_XX/{normal,abnormal}/*.wav)."
# )
# print(MF.groupby(["model_id", "label"]).size())

def discover_ids(cfg: Config) -> List[str]:
    # Dataset lokal tidak memiliki subfolder id_XX.
    # Gunakan satu ID virtual agar pipeline berikutnya tetap kompatibel.
    return ["id_00"]


def list_wavs(cfg: Config, mid: str, label: str) -> List[str]:
    # mid sengaja tidak dipakai: WAV berada langsung di Dataset/{normal,abnormal}.
    d = Path(cfg.dataset_root) / label
    return sorted(str(p) for p in d.glob("*.wav")) if d.exists() else []


def build_manifest(cfg: Config) -> pd.DataFrame:
    rows = []

    for mid in discover_ids(cfg):
        for lab in ("normal", "abnormal"):
            for p in list_wavs(cfg, mid, lab):
                rows.append({"path": p, "model_id": mid, "label": lab})

    return pd.DataFrame(rows, columns=["path", "model_id", "label"])


MF = build_manifest(CFG)

print(f"Manifest: {len(MF)} file. Machine IDs: {discover_ids(CFG)}")
assert len(MF) > 0, (
    "Manifest kosong. Pastikan CFG.dataset_root = 'Dataset' dan struktur foldernya "
    "Dataset/{normal,abnormal}/*.wav."
)

print(MF.groupby(["model_id", "label"]).size())

Manifest: 202 file. Machine IDs: ['id_00']
model_id  label   
id_00     abnormal    101
          normal      101
dtype: int64


## 5. Split Anti-Overfitting: Held-out Machine ID + Train/Val per Klip

- `holdout_machine_id` dikeluarkan total dari train/val — dipakai di akhir sebagai
  ukuran generalisasi ke unit yang benar-benar baru.
- Sisa ID di-subsample (maks `max_clips_per_class_per_id` klip per kelas per ID) lalu
  displit train/val **di level klip** (bukan window) supaya tidak ada kebocoran data.


In [ ]:
# assert CFG.holdout_machine_id in MF.model_id.unique(), \
#     f"holdout_machine_id={CFG.holdout_machine_id} tidak ada di manifest. ID tersedia: {discover_ids(CFG)}"

# seen = MF[MF.model_id != CFG.holdout_machine_id].copy()
# heldout = MF[MF.model_id == CFG.holdout_machine_id].copy()


# def subsample(df: pd.DataFrame, cap: int, seed: int) -> pd.DataFrame:
#     # Sengaja TIDAK pakai groupby(...).apply(lambda g: g.sample(...)) --
#     # sejak pandas >= 2.2 (default di pandas 3.x), .apply() pada GroupBy membuang
#     # kolom yang dipakai untuk group by dari hasilnya, sehingga baris berikutnya
#     # (stratify=df["label"]) akan meledak dengan KeyError. Loop manual ini aman
#     # di semua versi pandas.
#     parts = []
#     for (_mid, _lab), g in df.groupby(["model_id", "label"]):
#         parts.append(g.sample(n=min(len(g), cap), random_state=seed))
#     return pd.concat(parts, ignore_index=True) if parts else df.iloc[0:0]


# seen_sub = subsample(seen, CFG.max_clips_per_class_per_id, CFG.seed)
# heldout_sub = subsample(heldout, CFG.max_clips_per_class_per_id, CFG.seed)

# train_df, val_df = train_test_split(
#     seen_sub, test_size=CFG.train_val_split_frac,
#     stratify=seen_sub["label"], random_state=CFG.seed
# )

# print(f"Train clips: {len(train_df)} | Val clips: {len(val_df)} | "
#       f"Held-out ({CFG.holdout_machine_id}) clips: {len(heldout_sub)}")
# print("Train label dist:\n", train_df.label.value_counts())
# print("Val label dist:\n", val_df.label.value_counts())
# print("Held-out label dist:\n", heldout_sub.label.value_counts())

# assert set(train_df.path) & set(val_df.path) == set(), "LEAK: train/val overlap!"
# assert set(train_df.path) & set(heldout_sub.path) == set(), "LEAK: train/heldout overlap!"
# assert CFG.holdout_machine_id not in val_df.model_id.unique(), "LEAK: heldout ID muncul di val!"
# print("Tidak ada kebocoran data (train/val/held-out disjoint by clip dan by machine ID).")


def subsample(df: pd.DataFrame, cap: int, seed: int) -> pd.DataFrame:
    parts = []

    for (_mid, _lab), g in df.groupby(["model_id", "label"]):
        parts.append(g.sample(n=min(len(g), cap), random_state=seed))

    return pd.concat(parts, ignore_index=True) if parts else df.iloc[0:0]


# Dataset lokal hanya memiliki satu ID virtual, sehingga split dilakukan per clip.
all_sub = subsample(MF, CFG.max_clips_per_class_per_id, CFG.seed)

HOLDOUT_FRAC = 0.20

trainval_df, heldout_sub = train_test_split(
    all_sub,
    test_size=HOLDOUT_FRAC,
    stratify=all_sub["label"],
    random_state=CFG.seed,
)

train_df, val_df = train_test_split(
    trainval_df,
    test_size=CFG.train_val_split_frac,
    stratify=trainval_df["label"],
    random_state=CFG.seed,
)

print(
    f"Train clips: {len(train_df)} | "
    f"Val clips: {len(val_df)} | "
    f"Held-out test clips: {len(heldout_sub)}"
)
print("Train label dist:\n", train_df.label.value_counts())
print("Val label dist:\n", val_df.label.value_counts())
print("Held-out label dist:\n", heldout_sub.label.value_counts())

assert not (set(train_df.path) & set(val_df.path)), "LEAK: train/val overlap!"
assert not (set(train_df.path) & set(heldout_sub.path)), "LEAK: train/heldout overlap!"
assert not (set(val_df.path) & set(heldout_sub.path)), "LEAK: val/heldout overlap!"

print("Tidak ada kebocoran data antar train/val/held-out berdasarkan clip.")

Train clips: 128 | Val clips: 33 | Held-out test clips: 41
Train label dist:
 label
abnormal    64
normal      64
Name: count, dtype: int64
Val label dist:
 label
normal      17
abnormal    16
Name: count, dtype: int64
Held-out label dist:
 label
abnormal    21
normal      20
Name: count, dtype: int64
Tidak ada kebocoran data antar train/val/held-out berdasarkan clip.


## 6. Windowing untuk Training (non-overlap) + Augmentasi Kelas Minoritas

Beda dengan deployment (hop 1s, window saling overlap), di sini `hop_sec = window_sec`
supaya window antar-sampel lebih independen (tidak menduplikasi sinyal yang nyaris sama
berkali-kali, yang bisa membesar-besarkan efektif jumlah sampel secara semu).


In [ ]:
import soundfile as sf

TRAIN_CFG = replace(CFG, hop_sec=CFG.window_sec)


def build_window_dataset(
    df: pd.DataFrame,
    cfg: Config,
    train_cfg: Config,
    with_aug: bool = False,
    aug_rng: Optional[np.random.RandomState] = None,
) -> Tuple[List[np.ndarray], np.ndarray, List[str]]:
    X: List[np.ndarray] = []
    y: List[int] = []
    groups: List[str] = []
    for _, row in df.iterrows():
        audio, srate = sf.read(row["path"], dtype="float32", always_2d=True)
        waveform = torch.from_numpy(audio.T)
        if waveform.shape[0] > 1:
            waveform = waveform.mean(dim=0, keepdim=True)
        if srate != cfg.sr:
            waveform = torchaudio.functional.resample(waveform, srate, cfg.sr)
        raw = waveform.squeeze(0).numpy().astype(np.float64)

        xc = condition(raw, cfg.sr, cfg)
        label = 1 if row["label"] == "abnormal" else 0

        clips = [xc]
        if with_aug:
            assert aug_rng is not None, "aug_rng wajib diisi kalau with_aug=True"
            n_aug = cfg.aug_n_abnormal if label == 1 else cfg.aug_n_normal
            clips.extend(augment(xc, cfg.sr, cfg, aug_rng, n_aug))

        for clip in clips:
            for w in make_windows(clip, cfg.sr, train_cfg):
                X.append(w.astype(np.float32))
                y.append(label)
                groups.append(row["path"])
    return X, np.array(y, dtype=np.float32), groups


AUG_RNG = np.random.RandomState(CFG.seed + 1)

print("Membangun window set training...")
X_train, y_train, g_train = build_window_dataset(train_df, CFG, TRAIN_CFG, with_aug=True, aug_rng=AUG_RNG)
print(f"  train windows: {len(X_train)}  (pos={int(y_train.sum())}, neg={int((1-y_train).sum())}, "
      f"rasio neg:pos = {(1-y_train).sum()/max(y_train.sum(),1):.2f}:1)")

print("Membangun window set validasi...")
X_val, y_val, g_val = build_window_dataset(val_df, CFG, TRAIN_CFG, with_aug=False)
print(f"  val windows: {len(X_val)}  (pos={int(y_val.sum())}, neg={int((1-y_val).sum())})")

print("Membangun window set held-out...")
X_heldout, y_heldout, g_heldout = build_window_dataset(heldout_sub, CFG, TRAIN_CFG, with_aug=False)
print(f"  held-out windows: {len(X_heldout)}  (pos={int(y_heldout.sum())}, neg={int((1-y_heldout).sum())})")

assert len(X_train) > 0 and len(X_val) > 0 and len(X_heldout) > 0, \
    "Salah satu split menghasilkan 0 window -- cek Quality Gate / durasi klip."


Membangun window set training...
  train windows: 2240  (pos=1600, neg=640, rasio neg:pos = 0.40:1)
Membangun window set validasi...
  val windows: 165  (pos=80, neg=85)
Membangun window set held-out...
  held-out windows: 205  (pos=105, neg=100)


## 7. Dataset & Balanced Sampler

In [ ]:
class WindowDataset(Dataset):
    def __init__(self, X: List[np.ndarray], y: np.ndarray) -> None:
        self.X = X
        self.y = y

    def __len__(self) -> int:
        return len(self.X)

    def __getitem__(self, idx: int) -> Tuple[Tensor, Tensor]:
        return torch.from_numpy(self.X[idx]), torch.tensor(self.y[idx], dtype=torch.float32)


train_ds = WindowDataset(X_train, y_train)
val_ds = WindowDataset(X_val, y_val)
heldout_ds = WindowDataset(X_heldout, y_heldout)

# Balanced batch sampler: menyeimbangkan proporsi normal/abnormal per-batch (~50/50),
# MENGGANTIKAN pos_weight di loss (tidak dipakai bersamaan supaya tidak dobel-kompensasi).
# Catatan: karena keseimbangan kelas sudah jadi tugas sampler ini, FocalLoss di bawah
# sengaja dipakai dengan alpha netral (0.5) -- gamma-nya yang berperan menekan
# easy-examples, terlepas dari proporsi kelas di tiap batch.
class_count = np.array([(y_train == 0).sum(), (y_train == 1).sum()], dtype=np.float64)
class_weight = 1.0 / np.maximum(class_count, 1)
sample_weight = np.array([class_weight[int(label)] for label in y_train])
train_sampler = torch.utils.data.WeightedRandomSampler(
    weights=torch.from_numpy(sample_weight).double(),
    num_samples=len(y_train),
    replacement=True,
)

train_loader = DataLoader(train_ds, batch_size=CFG.finetune_batch_size, sampler=train_sampler)
val_loader = DataLoader(val_ds, batch_size=CFG.finetune_batch_size, shuffle=False)
heldout_loader = DataLoader(heldout_ds, batch_size=CFG.finetune_batch_size, shuffle=False)

print("DataLoader siap.")

DataLoader siap.


## 8. Muat BEATs Pretrained + Bungkus dengan Partial-Freeze Classifier Head

In [ ]:
from pathlib import Path
import subprocess
import sys

REPO_DIR = Path("third_party/unilm")
BEATS_DIR = REPO_DIR / "beats"
Path("third_party").mkdir(exist_ok=True)

if not BEATS_DIR.exists():
    # Jika folder unilm sudah ada dari clone gagal sebelumnya, jangan clone ulang
    # ke folder yang sama; tampilkan instruksi aman.
    if REPO_DIR.exists():
        raise RuntimeError(
            f"{REPO_DIR} sudah ada tetapi {BEATS_DIR} tidak ada. "
            "Hapus folder third_party/unilm yang merupakan hasil clone gagal, "
            "kemudian jalankan cell ini lagi."
        )

    result = subprocess.run(
        [
            "git", "clone",
            "--depth", "1",
            "--filter=blob:none",
            "--sparse",
            "https://github.com/microsoft/unilm.git",
            str(REPO_DIR),
        ],
        capture_output=True,
        text=True,
    )

    if result.returncode != 0:
        raise RuntimeError(f"Git clone gagal:\n{result.stderr}")

    result = subprocess.run(
        ["git", "-C", str(REPO_DIR), "sparse-checkout", "set", "beats"],
        capture_output=True,
        text=True,
    )

    if result.returncode != 0:
        raise RuntimeError(f"Sparse checkout gagal:\n{result.stderr}")

sys.path.insert(0, str(BEATS_DIR.resolve()))
from BEATs import BEATs, BEATsConfig

print("BEATs siap dari:", BEATS_DIR.resolve())

BEATs siap dari: C:\Users\Frederick\Documents\AIC\third_party\unilm\beats


In [ ]:
import torch

print("PyTorch:", torch.__version__)
print("CUDA build:", torch.version.cuda)
print("GPU tersedia:", torch.cuda.is_available())

if torch.cuda.is_available():
    print("GPU:", torch.cuda.get_device_name(0))
    device = torch.device("cuda:0")
else:
    device = torch.device("cpu")

print("device:", device)

PyTorch: 2.13.0+cpu
CUDA build: None
GPU tersedia: False
device: cpu


In [ ]:
from BEATs import BEATs, BEATsConfig

device: str = "cuda" if torch.cuda.is_available() else "cpu"
print("device:", device)
if device == "cpu":
    print("PERINGATAN: tidak ada GPU terdeteksi. Fine-tuning transformer di CPU sangat "
          "lambat -- disarankan Runtime > Change runtime type > GPU sebelum lanjut.")

checkpoint = torch.load(BASE_CHECKPOINT, map_location=device, weights_only=False)
beats_cfg = BEATsConfig(checkpoint['cfg'])
beats_model = BEATs(beats_cfg)
beats_model.load_state_dict(checkpoint['model'])
beats_model = beats_model.to(device)
print(f"BEATs pretrained dimuat. encoder_layers={beats_cfg.encoder_layers}, "
      f"encoder_embed_dim={beats_cfg.encoder_embed_dim}")


class BEATsClassifier(nn.Module):
    """Backbone BEATs + head klasifikasi ringan, dengan sebagian besar layer dibekukan."""

    def __init__(self, backbone: BEATs, n_unfrozen_layers: int, dropout: float = 0.3) -> None:
        super().__init__()
        self.backbone = backbone
        for p in self.backbone.parameters():
            p.requires_grad = False
        total_layers = len(self.backbone.encoder.layers)
        n_unfrozen = min(n_unfrozen_layers, total_layers)
        for layer in self.backbone.encoder.layers[total_layers - n_unfrozen:]:
            for p in layer.parameters():
                p.requires_grad = True
        for p in self.backbone.layer_norm.parameters():
            p.requires_grad = True
        self.dropout = nn.Dropout(dropout)
        self.head = nn.Linear(self.backbone.cfg.encoder_embed_dim, 1)

    def forward(self, wav: Tensor) -> Tensor:
        padding_mask = torch.zeros(wav.shape, dtype=torch.bool, device=wav.device)
        feats, _ = self.backbone.extract_features(wav, padding_mask=padding_mask)
        pooled = feats.mean(dim=1)
        pooled = self.dropout(pooled)
        return self.head(pooled).squeeze(-1)

    def embed(self, wav: Tensor) -> Tensor:
        """Embedding tanpa head -- ini yang dipakai lagi di pipeline deployment (Blok C-E).

        Dibungkus `torch.inference_mode()` (bukan `torch.no_grad()`) karena method ini
        murni dipakai untuk ekstraksi fitur read-only, tidak pernah di dalam training
        loop yang butuh backward -- `inference_mode` memberi overhead lebih rendah
        lewat C++ dispatch khusus di PyTorch modern.
        """
        with torch.inference_mode():
            padding_mask = torch.zeros(wav.shape, dtype=torch.bool, device=wav.device)
            feats, _ = self.backbone.extract_features(wav, padding_mask=padding_mask)
            emb = feats.mean(dim=1)
            return emb / (emb.norm(dim=-1, keepdim=True) + 1e-12)


model = BEATsClassifier(beats_model, CFG.n_unfrozen_layers, CFG.head_dropout).to(device)
n_trainable = sum(p.numel() for p in model.parameters() if p.requires_grad)
n_total = sum(p.numel() for p in model.parameters())
print(f"Trainable params: {n_trainable:,} / {n_total:,} ({100*n_trainable/n_total:.1f}%) "
      f"-- {CFG.n_unfrozen_layers} dari {len(beats_model.encoder.layers)} layer encoder dibuka.")
assert 0 < n_trainable < n_total, "Freeze/unfreeze logic salah (semua layer beku atau semua terbuka)!"

device: cpu
PERINGATAN: tidak ada GPU terdeteksi. Fine-tuning transformer di CPU sangat lambat -- disarankan Runtime > Change runtime type > GPU sebelum lanjut.
BEATs pretrained dimuat. encoder_layers=12, encoder_embed_dim=768
Trainable params: 14,182,441 / 90,312,561 (15.7%) -- 2 dari 12 layer encoder dibuka.


## 9. Loss Function — Focal Loss

**Kenapa Focal Loss, dan kenapa dikombinasikan seperti ini dengan sampler yang sudah ada:**

`WeightedRandomSampler` di sel sebelumnya sudah menangani *ketidakseimbangan kelas*
(memastikan tiap batch ~50/50 normal/abnormal). Itu **bukan** masalah yang sama dengan
*hard-example mining* — dalam batch yang sudah seimbang sekalipun, sebagian besar window
tetap "mudah" (jelas normal atau jelas abnormal) sementara sebagian kecil ambigu
(borderline, ter-augmentasi berat, atau derau tinggi). BCE memperlakukan keduanya setara;
Focal Loss men-downweight kontribusi window mudah lewat faktor `(1 - p_t)^gamma`, sehingga
gradien lebih terfokus ke window ambigu itu.

Karena keseimbangan kelas sudah jadi tugas sampler, `alpha` di implementasi ini **sengaja
dibiarkan netral (0.5)** secara default -- bukan dipakai untuk mengompensasi rasio
kelas lagi (itu akan dobel-kompensasi dengan sampler). Yang aktif bekerja di sini murni
`gamma`. Parameter ini tetap dibuat dapat dikonfigurasi (`CFG.focal_alpha`,
`CFG.focal_gamma`) kalau suatu saat sampler dinonaktifkan dan `alpha` perlu diaktifkan
kembali sebagai mekanisme pengimbang kelas.


In [ ]:
class FocalLoss(nn.Module):
    """Binary Focal Loss dari logits mentah (Lin et al., 2017), untuk klasifikasi biner.

    FL(p_t) = -alpha_t * (1 - p_t)^gamma * log(p_t)

    Implementasi numerically-stable: dihitung dari
    `binary_cross_entropy_with_logits` (bukan sigmoid + log manual), lalu
    `p_t = exp(-bce)` -- identik secara matematis dengan p jika target=1 dan
    (1-p) jika target=0, tanpa risiko log(0).
    """

    def __init__(self, alpha: float = 0.5, gamma: float = 2.0, reduction: str = "mean") -> None:
        super().__init__()
        if not 0.0 <= alpha <= 1.0:
            raise ValueError(f"alpha harus di rentang [0, 1], dapat {alpha}")
        if gamma < 0.0:
            raise ValueError(f"gamma harus >= 0, dapat {gamma}")
        if reduction not in ("mean", "sum", "none"):
            raise ValueError(f"reduction tidak dikenal: {reduction}")
        self.alpha = alpha
        self.gamma = gamma
        self.reduction = reduction

    def forward(self, logits: Tensor, targets: Tensor) -> Tensor:
        bce = F.binary_cross_entropy_with_logits(logits, targets, reduction="none")
        p_t = torch.exp(-bce)
        alpha_t = self.alpha * targets + (1.0 - self.alpha) * (1.0 - targets)
        focal_term = (1.0 - p_t).clamp(min=0.0) ** self.gamma
        loss = alpha_t * focal_term * bce
        if self.reduction == "mean":
            return loss.mean()
        if self.reduction == "sum":
            return loss.sum()
        return loss


criterion = FocalLoss(alpha=CFG.focal_alpha, gamma=CFG.focal_gamma)
print(f"Loss function: FocalLoss(alpha={CFG.focal_alpha}, gamma={CFG.focal_gamma}) "
      "-- alpha netral karena keseimbangan kelas sudah ditangani WeightedRandomSampler.")

Loss function: FocalLoss(alpha=0.5, gamma=2.0) -- alpha netral karena keseimbangan kelas sudah ditangani WeightedRandomSampler.


## 10. Learning Rate Scheduler — Linear Warmup + Cosine Annealing

Di-step **per batch** (bukan per epoch), dan berlaku proporsional untuk kedua param-group
(`lr_backbone` dan `lr_head`) sekaligus -- keduanya naik dari 0 selama warmup lalu meluruh
mengikuti kurva cosine menuju nol di akhir jadwal, tanpa mengubah rasio relatif
antar-keduanya (LR head tetap jauh lebih besar dari LR backbone di semua titik).

Dipilih dibanding `ReduceLROnPlateau` karena early stopping di notebook ini sudah memantau
`val_auc` secara eksplisit -- menambah `ReduceLROnPlateau` yang memantau metrik yang sama
akan membuat dua mekanisme bereaksi terhadap sinyal yang identik dengan cara yang saling
tumpang tindih. Warmup+cosine berjalan independen dari `val_auc`, murni fungsi dari
langkah training, sehingga perannya jelas terpisah dari early stopping.


In [ ]:
import torch._dynamo

def build_warmup_cosine_scheduler(
    optimizer: torch.optim.Optimizer,
    num_warmup_steps: int,
    num_training_steps: int,
    min_lr_ratio: float = 0.0,
) -> torch.optim.lr_scheduler.LambdaLR:
    """Linear warmup dari 0 -> lr dasar, lalu cosine decay -> `min_lr_ratio` * lr dasar.

    `min_lr_ratio` dijaga di sini (bukan langsung 0) supaya kalau training berhenti lebih
    awal karena early stopping, LR di beberapa epoch terakhir sebelum berhenti tidak jatuh
    ke nyaris nol -- yang bisa membuat epoch-epoch terakhir itu nyaris tidak belajar apa-apa.
    """
    def lr_lambda(current_step: int) -> float:
        if current_step < num_warmup_steps:
            return float(current_step) / float(max(1, num_warmup_steps))
        progress = float(current_step - num_warmup_steps) / float(
            max(1, num_training_steps - num_warmup_steps)
        )
        progress = min(progress, 1.0)
        cosine = 0.5 * (1.0 + np.cos(np.pi * progress))
        return min_lr_ratio + (1.0 - min_lr_ratio) * cosine

    return torch.optim.lr_scheduler.LambdaLR(optimizer, lr_lambda)


backbone_params = [p for n_, p in model.backbone.named_parameters() if p.requires_grad]
head_params = list(model.head.parameters())

optimizer = torch.optim.AdamW([
    {"params": backbone_params, "lr": CFG.finetune_lr_backbone},
    {"params": head_params, "lr": CFG.finetune_lr_head},
], weight_decay=CFG.finetune_weight_decay)

steps_per_epoch = len(train_loader)
total_steps = steps_per_epoch * CFG.finetune_epochs_max
warmup_steps = max(1, int(CFG.finetune_warmup_ratio * total_steps))
lr_scheduler = build_warmup_cosine_scheduler(optimizer, warmup_steps, total_steps, min_lr_ratio=0.05)

print(f"Scheduler siap: warmup {warmup_steps} step -> cosine annealing hingga {total_steps} step "
      f"total ({steps_per_epoch} step/epoch x {CFG.finetune_epochs_max} epoch maksimum). "
      "LR di-step tiap batch training, bukan tiap epoch.")

ModuleNotFoundError: No module named 'torch._dynamo'

## 11. Training Loop (Early Stopping, LR Berlapis + Warmup-Cosine, Focal Loss, Precision/Recall/F1)

`run_epoch` sekarang membungkus jalur non-training (`train_mode=False`) dengan
`torch.inference_mode()`. Ini juga memperbaiki inefisiensi di versi sebelumnya: dulu,
validasi tidak dibungkus context manager apa pun -- forward pass validasi tetap membangun
computation graph penuh (karena tidak ada `.backward()` yang dipanggil, graph itu memang
tidak dipakai, tapi tetap dialokasikan), memboroskan memori GPU tanpa manfaat. Dengan
`inference_mode()`, PyTorch tahu sejak awal graph itu tidak akan pernah dipakai untuk
autograd, sehingga alokasinya dilewati sepenuhnya.


In [ ]:
@dataclass
class EpochMetrics:
    loss: float
    auc: float
    precision: float
    recall: float
    f1: float

    def __str__(self) -> str:
        return (f"loss={self.loss:.4f} auc={self.auc:.3f} "
                f"precision={self.precision:.3f} recall={self.recall:.3f} f1={self.f1:.3f}")


def run_epoch(
    loader: DataLoader,
    train_mode: bool,
    scheduler: Optional[torch.optim.lr_scheduler.LambdaLR] = None,
    decision_threshold: float = 0.5,
) -> EpochMetrics:
    model.train(train_mode)
    total_loss = 0.0
    all_probs: List[float] = []
    all_labels: List[float] = []

    grad_context = torch.enable_grad() if train_mode else torch.inference_mode()
    with grad_context:
        for wav, label in loader:
            wav, label = wav.to(device), label.to(device)
            if train_mode:
                optimizer.zero_grad(set_to_none=True)
            logits = model(wav)
            loss = criterion(logits, label)
            if train_mode:
                loss.backward()
                optimizer.step()
                if scheduler is not None:
                    scheduler.step()
            total_loss += loss.item() * len(label)
            probs = torch.sigmoid(logits).detach().cpu()
            all_probs.extend(probs.numpy().tolist())
            all_labels.extend(label.detach().cpu().numpy().tolist())

    avg_loss = total_loss / len(loader.dataset)
    preds = [1 if p >= decision_threshold else 0 for p in all_probs]
    try:
        auc = roc_auc_score(all_labels, all_probs)
    except ValueError:
        auc = float("nan")
    precision = precision_score(all_labels, preds, zero_division=0)
    recall = recall_score(all_labels, preds, zero_division=0)
    f1 = f1_score(all_labels, preds, zero_division=0)
    return EpochMetrics(loss=avg_loss, auc=auc, precision=precision, recall=recall, f1=f1)


# sanity-check: pastikan sampler benar-benar menyeimbangkan batch (~50/50)
_check_labels: List[float] = []
for _, label in train_loader:
    _check_labels.extend(label.numpy().tolist())
    if len(_check_labels) >= 200:
        break
_bal = float(np.mean(_check_labels))
print(f"Proporsi kelas abnormal di batch ter-sampling (target ~0.5): {_bal:.3f}")
assert 0.3 < _bal < 0.7, f"Sampler tidak menyeimbangkan kelas dengan baik (proporsi={_bal:.3f})."

history: Dict[str, List[float]] = {
    "train_loss": [], "train_auc": [], "train_precision": [], "train_recall": [], "train_f1": [],
    "val_loss": [], "val_auc": [], "val_precision": [], "val_recall": [], "val_f1": [],
    "lr_backbone": [], "lr_head": [],
}
best_val_auc: float = -1.0
best_state: Optional[Dict[str, Tensor]] = None
patience_ctr: int = 0

print("\nMulai training...")
for epoch in range(CFG.finetune_epochs_max):
    tr = run_epoch(train_loader, train_mode=True, scheduler=lr_scheduler)
    va = run_epoch(val_loader, train_mode=False)

    history["train_loss"].append(tr.loss); history["train_auc"].append(tr.auc)
    history["train_precision"].append(tr.precision); history["train_recall"].append(tr.recall)
    history["train_f1"].append(tr.f1)
    history["val_loss"].append(va.loss); history["val_auc"].append(va.auc)
    history["val_precision"].append(va.precision); history["val_recall"].append(va.recall)
    history["val_f1"].append(va.f1)
    history["lr_backbone"].append(optimizer.param_groups[0]["lr"])
    history["lr_head"].append(optimizer.param_groups[1]["lr"])

    print(f"  epoch {epoch+1:02d} [train] {tr}")
    print(f"              [val]   {va}  | lr_backbone={optimizer.param_groups[0]['lr']:.2e} "
          f"lr_head={optimizer.param_groups[1]['lr']:.2e}")

    if va.auc > best_val_auc:
        best_val_auc = va.auc
        best_state = copy.deepcopy(model.backbone.state_dict())
        patience_ctr = 0
    else:
        patience_ctr += 1
        if patience_ctr >= CFG.finetune_patience:
            print(f"  Early stopping (val_auc tidak membaik {CFG.finetune_patience} epoch berturut-turut).")
            break

    if device == "cuda":
        torch.cuda.empty_cache()

assert best_state is not None, "Tidak ada checkpoint terbaik tersimpan!"
model.backbone.load_state_dict(best_state)
print(f"\nBest val AUC: {best_val_auc:.4f} -- backbone dikembalikan ke state terbaik (bukan epoch terakhir).")

## 12. Evaluasi Generalisasi di Held-out Machine ID (Unit yang Tak Pernah Dilihat)

Ini ukuran overfitting yang paling jujur: kalau `train_auc`/`val_auc` tinggi tapi
`held-out AUC` jauh lebih rendah, berarti model menghafal karakteristik akustik unit
mesin yang dilihat saat training, bukan belajar pola "normal vs abnormal" yang general.


In [ ]:
_ = None
heldout_metrics_head = run_epoch(heldout_loader, train_mode=False)
print(f"Held-out (random 20% clip split) -- classifier head: {heldout_metrics_head}")


def compute_normal_centroid(embed_fn: Callable[[Tensor], Tensor], loader: DataLoader) -> Tensor:
    """Rata-rata embedding kelas normal di training set -- baseline "sehat"."""
    embs: List[Tensor] = []
    with torch.inference_mode():
        for wav, label in loader:
            mask = label.numpy() == 0
            if mask.sum() == 0:
                continue
            e = embed_fn(wav[torch.from_numpy(mask)].to(device))
            embs.append(e)
    return torch.cat(embs, dim=0).mean(dim=0)


def compute_cosine_distances(
    embed_fn: Callable[[Tensor], Tensor], loader: DataLoader, centroid: Tensor
) -> Tuple[List[float], List[float]]:
    """Jarak kosinus tiap window uji ke centroid normal, plus label groundtruth-nya."""
    dists: List[float] = []
    labels: List[float] = []
    with torch.inference_mode():
        for wav, label in loader:
            e = embed_fn(wav.to(device))
            d = 1 - F.cosine_similarity(e, centroid.unsqueeze(0).expand(e.shape[0], -1))
            dists.extend(d.cpu().numpy().tolist())
            labels.extend(label.numpy().tolist())
    return dists, labels


# Evaluasi ala pipeline deployment (Blok C): jarak kosinus embedding ke centroid data
# normal, TANPA head klasifikasi -- ini yang relevan untuk dipakai lagi di v4.
model.eval()
centroid = compute_normal_centroid(model.embed, train_loader)
dists_ft, labels_ho = compute_cosine_distances(model.embed, heldout_loader, centroid)
auc_embedding_finetuned = roc_auc_score(labels_ho, dists_ft)
print(f"Held-out (random 20% clip split) AUC -- embedding-centroid (backbone FINE-TUNED): "
      f"{auc_embedding_finetuned:.4f}")

# Baseline pembanding: embedding-centroid pakai backbone ORIGINAL (belum di-finetune),
# supaya jelas terlihat fine-tuning ini benar-benar menambah nilai atau tidak.
orig_beats = BEATs(beats_cfg)
orig_ckpt = torch.load(BASE_CHECKPOINT, map_location=device, weights_only=False)
orig_beats.load_state_dict(orig_ckpt["model"])
orig_beats = orig_beats.to(device).eval()


def embed_orig(wav: Tensor) -> Tensor:
    with torch.inference_mode():
        padding_mask = torch.zeros(wav.shape, dtype=torch.bool, device=wav.device)
        feats, _ = orig_beats.extract_features(wav, padding_mask=padding_mask)
        emb = feats.mean(dim=1)
        return emb / (emb.norm(dim=-1, keepdim=True) + 1e-12)


centroid_orig = compute_normal_centroid(embed_orig, train_loader)
dists_orig, labels_ho2 = compute_cosine_distances(embed_orig, heldout_loader, centroid_orig)
auc_embedding_original = roc_auc_score(labels_ho2, dists_orig)

print("\n" + "=" * 60)
print("RINGKASAN GENERALISASI -- held-out: random 20% clip split (dataset lokal tidak memiliki machine ID)")
print("=" * 60)
print(f"  Zero-shot (BEATs pretrained asli, TANPA fine-tune) : AUC = {auc_embedding_original:.4f}")
print(f"  Fine-tuned (backbone hasil training ini)            : AUC = {auc_embedding_finetuned:.4f}")
delta = auc_embedding_finetuned - auc_embedding_original
print(f"  Selisih                                             : {delta:+.4f}")
if delta > 0:
    print("  -> Fine-tuning meningkatkan kualitas embedding untuk unit yang belum pernah dilihat.")
else:
    print("  -> Fine-tuning TIDAK terbukti membantu di unit held-out ini -- pertimbangkan lebih banyak "
          "data, lebih sedikit layer dibuka, atau early stopping lebih ketat.")

if device == "cuda":
    torch.cuda.empty_cache()

## 13. Simpan Checkpoint (CPU-Safe, Drop-in Compatible dengan `acousticare_v4_beats.ipynb`)

Checkpoint disimpan dengan tensor dipindah **eksplisit** ke CPU dan di-cast ke `float32`
sebelum `torch.save` -- bukan mengandalkan `map_location` saat *load* nanti. Alasannya:
kalau notebook ini dijalankan di GPU, `state_dict()` defaultnya berisi tensor CUDA;
`map_location` di sisi pemuat memang seharusnya tetap menanganinya dengan benar, tapi itu
menaruh syarat kebenaran pada kode *pemanggil* (mis. server FastAPI) untuk selalu ingat
mengatur `map_location`. Dengan membuat file checkpoint itu sendiri sudah "netral device"
sejak disimpan, environment CPU-only yang memuatnya dengan cara paling naif sekalipun
(`torch.load(path)` tanpa argumen tambahan) tetap akan bekerja tanpa error unpickling
atau device mismatch. Verifikasi di bawah membuktikan ini secara eksplisit, bukan sekadar
diasumsikan.


In [ ]:
def save_checkpoint_cpu_compatible(
    backbone_state_dict: Dict[str, Tensor],
    cfg_dict: dict,
    path: str,
) -> None:
    """Simpan checkpoint yang dijamin bisa di-load di lingkungan CPU-only (mis. server
    inferensi FastAPI) tanpa unpickling error atau device mismatch.

    - Setiap tensor dipindah eksplisit ke CPU dan di-cast ke float32.
    - `cfg` disimpan sebagai dict Python murni (bukan objek `BEATsConfig`), supaya proses
      unpickling di sisi pemuat tidak bergantung pada tersedianya definisi class yang
      identik persis di environment tujuan.
    """
    state_dict_cpu: Dict[str, Tensor] = {
        k: v.detach().to("cpu", dtype=torch.float32).contiguous()
        for k, v in backbone_state_dict.items()
    }
    ckpt = {"cfg": dict(cfg_dict), "model": state_dict_cpu}
    torch.save(ckpt, path)


OUTPUT_CHECKPOINT = "BEATs_finetuned_fan.pt"
save_checkpoint_cpu_compatible(model.backbone.state_dict(), beats_cfg.__dict__, OUTPUT_CHECKPOINT)
print(f"Checkpoint fine-tuned disimpan: {OUTPUT_CHECKPOINT}")

# Verifikasi: reload di CPU murni (device dipaksa "cpu", terlepas dari device training),
# meniru persis kondisi runtime FastAPI CPU-only -- lalu cek eksplisit tiap tensor.
_reloaded = torch.load(OUTPUT_CHECKPOINT, map_location="cpu", weights_only=False)
_reload_cfg = BEATsConfig(_reloaded["cfg"])
_reload_model = BEATs(_reload_cfg)
_reload_model.load_state_dict(_reloaded["model"])
_reload_model = _reload_model.to("cpu").eval()

_devices = {v.device.type for v in _reloaded["model"].values()}
_dtypes = {v.dtype for v in _reloaded["model"].values()}
assert _devices == {"cpu"}, f"Checkpoint mengandung tensor non-CPU: {_devices}"
assert _dtypes == {torch.float32}, f"Checkpoint mengandung dtype selain float32: {_dtypes}"
print(f"Verifikasi reload OK -- {len(_reloaded['model'])} tensor, semuanya di CPU dan float32. "
      "File ini aman dimuat langsung di lingkungan FastAPI CPU-only tanpa map_location eksplisit, "
      "dan bisa dipakai sebagai CHECKPOINT_FILENAME di acousticare_v4_beats.ipynb.")

try:
    from google.colab import files
    files.download(OUTPUT_CHECKPOINT)
except Exception:
    print("(Bukan di Colab / auto-download tidak tersedia -- ambil file secara manual dari "
          f"file browser sebelah kiri: {OUTPUT_CHECKPOINT})")

## 14. Plot: Kurva Training, Metrik, dan Jadwal LR

In [ ]:
import matplotlib.pyplot as plt

fig, axes = plt.subplots(2, 2, figsize=(13, 9))

ax = axes[0, 0]
ax.plot(history["train_loss"], marker="o", label="train_loss")
ax.plot(history["val_loss"], marker="o", label="val_loss")
ax.set_xlabel("Epoch"); ax.set_ylabel("Focal Loss")
ax.set_title("Kurva Loss (val naik sementara train turun = tanda overfitting)")
ax.legend()

ax = axes[0, 1]
ax.plot(history["train_auc"], marker="o", label="train_auc")
ax.plot(history["val_auc"], marker="o", label="val_auc")
ax.axhline(auc_embedding_finetuned, color="red", ls="--",
           label=f"held-out AUC ({auc_embedding_finetuned:.3f})")
ax.axhline(auc_embedding_original, color="gray", ls=":",
           label=f"held-out AUC zero-shot ({auc_embedding_original:.3f})")
ax.set_xlabel("Epoch"); ax.set_ylabel("AUC")
ax.set_title("Kurva AUC vs Generalisasi ke Unit Baru")
ax.legend()

ax = axes[1, 0]
ax.plot(history["val_precision"], marker="o", label="val_precision")
ax.plot(history["val_recall"], marker="o", label="val_recall")
ax.plot(history["val_f1"], marker="o", label="val_f1")
ax.set_xlabel("Epoch"); ax.set_ylabel("Skor")
ax.set_title("Precision / Recall / F1 pada threshold 0.5 (validasi)")
ax.legend()

ax = axes[1, 1]
ax.plot(history["lr_backbone"], marker=".", label="lr_backbone")
ax.plot(history["lr_head"], marker=".", label="lr_head")
ax.set_yscale("log")
ax.set_xlabel("Epoch"); ax.set_ylabel("Learning rate (skala log)")
ax.set_title("Jadwal LR -- Warmup Linear -> Cosine Annealing")
ax.legend()

plt.tight_layout()
plt.show()

## 15. Langkah Selanjutnya

1. Unduh `BEATs_finetuned_fan.pt` (otomatis ke-download di Colab lewat sel 13, atau ambil
   manual dari file browser).
2. Buka `acousticare_v4_beats.ipynb`, di cell "Muat Model BEATs", ganti
   `CHECKPOINT_FILENAME = "BEATs_iter3plus_AS2M_pretrained.pt"` menjadi
   `CHECKPOINT_FILENAME = "BEATs_finetuned_fan.pt"` (upload file itu ke sesi Colab-nya).
3. Sisa pipeline (Quality Gate → Memory Bank/LOO → threshold dinamis → Health Card)
   **tidak perlu diubah** — hanya backbone embedding-nya yang kini teradaptasi ke domain
   suara fan MIMII, bukan lagi AudioSet umum.
4. Untuk deployment CPU-only (mis. FastAPI): checkpoint di sel 13 sudah diverifikasi
   otomatis 100% berisi tensor CPU + float32, jadi `torch.load("BEATs_finetuned_fan.pt")`
   di server tanpa GPU akan langsung berhasil tanpa perlu argumen `map_location` khusus.

**Kalau hasil di Bagian 12 menunjukkan held-out AUC fine-tuned lebih rendah atau setara
dengan zero-shot**, itu sinyal jujur bahwa fine-tuning belum membantu untuk konfigurasi
ini — coba naikkan `max_clips_per_class_per_id` (lebih banyak data), turunkan
`n_unfrozen_layers` ke 1 (lebih konservatif), atau perpanjang `finetune_patience` sedikit.
Jangan menaikkan `n_unfrozen_layers` atau menurunkan `finetune_weight_decay` sebagai
reaksi pertama — itu justru berlawanan arah dengan tujuan "jangan overfit". Kalau
precision tinggi tapi recall rendah (atau sebaliknya) di Bagian 11-12, pertimbangkan
menggeser `decision_threshold` di `run_epoch` alih-alih mengubah `focal_alpha` --
threshold adalah keputusan operasional pasca-training, sementara `focal_alpha` mengubah
apa yang dipelajari model saat training.
